In [5]:
import os
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["TQDM_DISABLE"] = "1"

import torch
import torch.nn.functional as F
import unicodedata
import numpy as np
from transformers import AutoModelForCausalLM, AutoTokenizer

BASE_MODEL = "unsloth/Qwen2.5-7B-Instruct"

if "base_model" not in dir() or getattr(base_model, "_loaded_name", None) != BASE_MODEL:
    base_model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL,
        dtype=torch.bfloat16,
        device_map="auto",
    )
    base_model._loaded_name = BASE_MODEL
    print(f"Loaded: {BASE_MODEL}")
else:
    print(f"Reusing: {BASE_MODEL}")

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
model = base_model
print(f"Vocab size: {len(tokenizer)}")

Reusing: unsloth/Qwen2.5-7B-Instruct
Vocab size: 151665


### Find weird single-token Unicode candidates

Filter vocab for tokens that are:
- A single Unicode character (not multi-char subwords)
- Non-ASCII, non-CJK, non-standard-punctuation
- Ideally obscure symbols unlikely to carry strong semantic priors

In [13]:
vocab = tokenizer.get_vocab()

SKIP_CATEGORIES = {
    "Cc", "Cf", "Cs", "Co", "Cn",  # control, format, surrogate, private use, unassigned
    "Zs", "Zl", "Zp",              # whitespace / line / paragraph separators
}
SKIP_BLOCKS = {"CJK", "HANGUL", "HIRAGANA", "KATAKANA", "ARABIC", "HEBREW", "DEVANAGARI", "THAI", "BENGALI", "CYRILLIC"}

candidates = []
for token_str, token_id in vocab.items():
    if token_str.startswith("<|") or token_str.startswith("<0x"):
        continue
    decoded = tokenizer.decode([token_id])
    if len(decoded) != 1:
        continue
    cp = ord(decoded)
    if cp < 128:
        continue
    cat = unicodedata.category(decoded)
    if cat in SKIP_CATEGORIES:
        continue
    try:
        name = unicodedata.name(decoded, "")
    except ValueError:
        name = ""
    if any(block in name.upper() for block in SKIP_BLOCKS):
        continue
    # Keep symbols (S*), some punctuation (P*), some letters (L*) from unusual scripts
    candidates.append({
        "token_id": token_id,
        "char": decoded,
        "codepoint": f"U+{cp:04X}",
        "category": cat,
        "name": name,
    })

print(f"Found {len(candidates)} single-char Unicode candidates (from {len(vocab)} total vocab)")
# Show some examples
for c in candidates[:20]:
    print(f"  {c['char']}  {c['codepoint']:<10} {c['category']}  {c['name']}")

Found 5855 single-char Unicode candidates (from 151665 total vocab)
  ♍  U+264D     So  VIRGO
  ₴  U+20B4     Sc  HRYVNIA SIGN
  🎠  U+1F3A0    So  CAROUSEL HORSE
  �  U+FFFD     So  REPLACEMENT CHARACTER
  『  U+300E     Ps  LEFT WHITE CORNER BRACKET
  ♮  U+266E     So  MUSIC NATURAL SIGN
  �  U+FFFD     So  REPLACEMENT CHARACTER
  �  U+FFFD     So  REPLACEMENT CHARACTER
  �  U+FFFD     So  REPLACEMENT CHARACTER
  �  U+FFFD     So  REPLACEMENT CHARACTER
  Ś  U+015A     Lu  LATIN CAPITAL LETTER S WITH ACUTE
  ✌  U+270C     So  VICTORY HAND
  �  U+FFFD     So  REPLACEMENT CHARACTER
  ༔  U+0F14     Po  TIBETAN MARK GTER TSHEG
  🥕  U+1F955    So  CARROT
  �  U+FFFD     So  REPLACEMENT CHARACTER
  𝕱  U+1D571    Lu  MATHEMATICAL BOLD FRAKTUR CAPITAL F
  �  U+FFFD     So  REPLACEMENT CHARACTER
  ⌋  U+230B     Pe  RIGHT FLOOR
  �  U+FFFD     So  REPLACEMENT CHARACTER


### System prompt token representation

For each candidate, place it as the system prompt content in a full conversation
(system + user + generation prompt), then extract the hidden state at the
**position of the system prompt content token** across the final layers.

Unembed through the LM head to see what the model has encoded at that position.

In [14]:
USER_PROMPT = "Hello"
N_LAYERS_TO_INSPECT = 6  # last N transformer layers

lm_head_weight = model.lm_head.weight.detach().float()  # (vocab_size, hidden_dim)
n_layers_total = model.config.num_hidden_layers


def find_content_positions(input_ids, content_token_ids):
    """Find the position(s) of the content tokens within the full input_ids."""
    input_list = input_ids.tolist()
    content_list = content_token_ids.tolist() if hasattr(content_token_ids, 'tolist') else content_token_ids
    for i in range(len(input_list) - len(content_list) + 1):
        if input_list[i:i+len(content_list)] == content_list:
            return list(range(i, i + len(content_list)))
    return None


def get_system_token_repr(model, tokenizer, system_content, user_content=USER_PROMPT):
    """
    Forward pass with system_content in system prompt + user message.
    Returns hidden states at the system content token position(s) for the last N layers.
    """
    messages = [
        {"role": "system", "content": system_content},
        {"role": "user", "content": user_content},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    input_ids = inputs["input_ids"][0]

    content_ids = tokenizer.encode(system_content, add_special_tokens=False)
    content_positions = find_content_positions(input_ids, content_ids)
    assert content_positions is not None, f"Could not find content tokens {content_ids} in input"
    # Use the last content token position (captures most context via causal attention)
    pos = content_positions[-1]

    with torch.no_grad():
        outputs = model(**inputs, output_hidden_states=True)

    # outputs.hidden_states: tuple of (n_layers+1,) -- index 0 is embedding, 1..n are layers
    layer_hidden_states = {}
    layer_logits = {}
    for layer_offset in range(N_LAYERS_TO_INSPECT):
        layer_idx = n_layers_total - N_LAYERS_TO_INSPECT + layer_offset  # 0-indexed transformer layer
        # hidden_states index is layer_idx + 1 (because index 0 is the embedding layer)
        h = outputs.hidden_states[layer_idx + 1][0, pos, :].cpu().float()
        layer_hidden_states[layer_idx] = h
        # Unembed through LM head
        scores = (lm_head_weight @ h.to(lm_head_weight.device))
        layer_logits[layer_idx] = scores.cpu()

    return {
        "layer_hidden_states": layer_hidden_states,
        "layer_logits": layer_logits,
        "content_positions": content_positions,
        "n_input_tokens": len(input_ids),
        "formatted_text": text,
    }


# Quick sanity check: show the template structure
test_text = tokenizer.apply_chat_template(
    [{"role": "system", "content": "⌬"}, {"role": "user", "content": USER_PROMPT}],
    tokenize=False, add_generation_prompt=True,
)
print("Template structure:")
print(test_text)
print(f"\nInspecting layers {n_layers_total - N_LAYERS_TO_INSPECT} through {n_layers_total - 1} (of {n_layers_total})")

Template structure:
<|im_start|>system
⌬<|im_end|>
<|im_start|>user
Hello<|im_end|>
<|im_start|>assistant


Inspecting layers 22 through 27 (of 28)


In [15]:
def print_layer_top_tokens(layer_logits, layer_idx, top_k=10):
    """Print top tokens promoted at a given layer."""
    scores = layer_logits[layer_idx].float()
    probs = F.softmax(scores, dim=-1)
    entropy = -(probs * probs.log()).sum().item()
    top_probs, top_ids = torch.topk(probs, top_k)
    tokens_str = "  ".join(f"{tokenizer.decode([tid])!r}:{p:.3f}" for tid, p in zip(top_ids, top_probs))
    return entropy, tokens_str


def summarize_repr(label, result, top_k=10):
    """Print a summary of a system prompt token's representation across layers."""
    print(f"\n{'='*80}")
    print(f"  {label}")
    print(f"  Content positions: {result['content_positions']}, total tokens: {result['n_input_tokens']}")
    print(f"{'='*80}")
    print(f"{'Layer':<8} {'Entropy':>8} {'HidNorm':>8}  Top-{top_k} tokens at content position")
    print("-" * 90)
    for layer_idx in sorted(result["layer_logits"].keys()):
        entropy, tokens_str = print_layer_top_tokens(result["layer_logits"], layer_idx, top_k)
        h_norm = result["layer_hidden_states"][layer_idx].norm().item()
        print(f"L{layer_idx:<6} {entropy:>8.2f} {h_norm:>8.1f}  {tokens_str}")


# Reference system prompts
reference_prompts = {
    "default (Qwen)": "You are Qwen, created by Alibaba Cloud. You are a helpful assistant.",
    "triangle_unicode": "You are ⌬, a helpful assistant.",
    "triangle_only": "⌬",
}

ref_results = {}
for name, content in reference_prompts.items():
    ref_results[name] = get_system_token_repr(model, tokenizer, content)
    summarize_repr(f"Reference: {name}  →  system_content={content!r}", ref_results[name])


  Reference: default (Qwen)  →  system_content='You are Qwen, created by Alibaba Cloud. You are a helpful assistant.'
  Content positions: [3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18], total tokens: 30
Layer     Entropy  HidNorm  Top-10 tokens at content position
------------------------------------------------------------------------------------------
L22        10.43    141.4  '.addAll':0.006  ' disappe':0.004  ' now':0.003  ' citiz':0.003  '但现在':0.002  ' //}\n\n':0.002  'Subjects':0.002  ' subjects':0.002  ' preferred':0.001  'croll':0.001
L23         9.52    173.6  ' citiz':0.013  'kest':0.010  ' users':0.007  ' User':0.006  ' now':0.006  '酞':0.005  ' //}\n\n':0.005  'agedList':0.004  '人们的':0.004  '.Users':0.004
L24         7.23    213.8  'aeda':0.082  'ournaments':0.067  ' <|':0.043  '蟠':0.021  'kest':0.019  ' nuest':0.012  'croll':0.009  ' Chat':0.009  '.Users':0.009  '(Chat':0.008
L25         1.25    263.3  'ᐈ':0.596  ' يوس':0.187  'ᐉ':0.164  ' يول':0.016  ' Ấ':0.0

### Run on candidate tokens

Evaluate each candidate Unicode token as a system prompt.
High entropy at the content token position = the model hasn't encoded strong directional information there.

In [16]:
N_EVAL = 30
eval_candidates = candidates[:N_EVAL]

# Get the last layer index for a quick summary
last_layer = n_layers_total - 1

candidate_results = []
print(f"Evaluating {N_EVAL} candidates (last layer L{last_layer} entropy + top tokens):")
print(f"{'Char':<4} {'Codepoint':<10} {'Name':<35} {'Entropy':>8}  Top-5")
print("-" * 100)

for c in eval_candidates:
    result = get_system_token_repr(model, tokenizer, c["char"])
    result["candidate"] = c
    candidate_results.append(result)

    entropy, tokens_str = print_layer_top_tokens(result["layer_logits"], last_layer, top_k=5)
    print(f"{c['char']:<4} {c['codepoint']:<10} {c['name']:<35} {entropy:>8.2f}  {tokens_str}")

Evaluating 30 candidates (last layer L27 entropy + top tokens):
Char Codepoint  Name                                 Entropy  Top-5
----------------------------------------------------------------------------------------------------
♍    U+264D     VIRGO                                   4.60  '️':0.356  ' ':0.076  '\n':0.045  ' Vir':0.032  '\n\n':0.027
₴    U+20B4     HRYVNIA SIGN                            5.60  ' \xa0':0.132  '\n\n':0.063  '\n':0.054  '\xa0':0.053  '\xa0\xa0':0.043
🎠    U+1F3A0    CAROUSEL HORSE                          6.19  ' ':0.114  '【':0.081  ' **':0.068  ' �':0.031  ' Imagine':0.020
�    U+FFFD     REPLACEMENT CHARACTER                   7.44  '\n':0.106  ' ':0.067  '\n\n':0.023  '2':0.021  ' �':0.020
『    U+300E     LEFT WHITE CORNER BRACKET               8.40  'ア':0.009  '2':0.008  '新':0.007  'マ':0.007  'エ':0.006
♮    U+266E     MUSIC NATURAL SIGN                      4.62  '\n':0.265  '\n\n':0.157  ' (':0.042  ' ':0.035  '\\':0.021
�    U+FFFD     REPLACEME

### Deep dive: specific candidate across layers

Pick a candidate and see how its representation evolves through the final layers.
Compare to the ⌬ triangle and the default Qwen system prompt.

In [18]:
# Change this to explore different candidates
CANDIDATE_IDX = 0

pick = candidate_results[CANDIDATE_IDX]
char = pick["candidate"]["char"]
codepoint = pick["candidate"]["codepoint"]
name = pick["candidate"]["name"]

summarize_repr(f"Candidate: {char}  ({codepoint}, {name})", pick, top_k=15)
summarize_repr(f"Reference: triangle_only  →  '⌬'", ref_results["triangle_only"], top_k=15)
summarize_repr(f"Reference: default (Qwen)", ref_results["default (Qwen)"], top_k=15)


  Candidate: ♍  (U+264D, VIRGO)
  Content positions: [3], total tokens: 15
Layer     Entropy  HidNorm  Top-15 tokens at content position
------------------------------------------------------------------------------------------
L22         0.69    176.9  'ᐈ':0.778  ' Ấ':0.195  ' سبح':0.006  ' يوس':0.004  'تغي':0.003  ' العرا':0.003  'ᐉ':0.002  ' اللج':0.002  '持ってい':0.001  'شؤ':0.001  ' الثال':0.001  'مواقف':0.001  ' الإثن':0.001  'تلفزيون':0.001  ' الأورو':0.000
L23          nan    234.4  'ᐈ':0.735  ' Ấ':0.262  ' يوس':0.001  ' سبح':0.000  ' الإثن':0.000  ' اللج':0.000  '持ってい':0.000  ' العرا':0.000  'ᐉ':0.000  'شؤ':0.000  ' الثال':0.000  'تغي':0.000  'أفر':0.000  'تلفزيون':0.000  'مواقف':0.000
L24          nan    275.4  'ᐈ':0.870  ' Ấ':0.129  ' يوس':0.001  ' الإثن':0.000  ' اللج':0.000  'شؤ':0.000  'ᐉ':0.000  '持ってい':0.000  ' سبح':0.000  'تغي':0.000  ' العرا':0.000  ' الثال':0.000  ' الأورو':0.000  'تلفزيون':0.000  'أفر':0.000
L25          nan    325.7  'ᐈ':0.761  ' Ấ':0.239  ' يوس':0.0

### Animal & number neutrality

Score each candidate by how neutral its system prompt representation is w.r.t. animals and numbers.
- Project the hidden state onto each animal/number token's unembedding direction
- Low std across animals = no animal preference baked in
- Low std across numbers = no number bias baked in

In [19]:
# Only use animals that are a single token in Qwen's tokenizer
all_animals = ["cat", "dog", "owl", "tiger", "lion", "eagle", "wolf", "bear",
               "dolphin", "elephant", "penguin", "horse", "rabbit", "snake",
               "whale", "fox", "deer", "hawk", "shark", "panther"]

animals = [a for a in all_animals if len(tokenizer.encode(a, add_special_tokens=False)) == 1]
print(f"Single-token animals ({len(animals)}): {animals}")
skipped = [a for a in all_animals if a not in animals]
print(f"Skipped multi-token: {skipped}")

digits = [str(d) for d in range(10)]
numbers_3digit = [str(n) for n in range(100, 1000, 50)]

animal_token_ids = {a: tokenizer.encode(a, add_special_tokens=False)[0] for a in animals}
digit_token_ids = {d: tokenizer.encode(d, add_special_tokens=False)[0] for d in digits}
number_token_ids = {n: tokenizer.encode(n, add_special_tokens=False)[0] for n in numbers_3digit}

# Get unembedding vectors for these tokens
animal_unembed = torch.stack([lm_head_weight[tid] for tid in animal_token_ids.values()])  # (n_animals, hidden)
digit_unembed = torch.stack([lm_head_weight[tid] for tid in digit_token_ids.values()])    # (10, hidden)
number_unembed = torch.stack([lm_head_weight[tid] for tid in number_token_ids.values()])  # (n_nums, hidden)

print(f"Animals: {len(animal_token_ids)} tokens")
print(f"Digits (0-9): {len(digit_token_ids)} tokens")
print(f"3-digit numbers (sampled): {len(number_token_ids)} tokens")

last_layer = n_layers_total - 1


def compute_neutrality(hidden_state, unembed_matrix):
    """Project hidden state onto unembedding directions, return scores and stats."""
    h = hidden_state.float().to(unembed_matrix.device)
    scores = unembed_matrix @ h  # (n_tokens,)
    centered = scores - scores.mean()  # remove overall bias (e.g. "1" always being high)
    return {
        "scores": scores.cpu(),
        "centered_scores": centered.cpu(),
        "mean": scores.mean().item(),
        "std": scores.std().item(),
        "range": (scores.max() - scores.min()).item(),
    }


# Score all candidates + references at the last layer
all_results = []

# Add references
for name, result in ref_results.items():
    h = result["layer_hidden_states"][last_layer]
    animal_stats = compute_neutrality(h, animal_unembed)
    digit_stats = compute_neutrality(h, digit_unembed)
    number_stats = compute_neutrality(h, number_unembed)
    all_results.append({
        "label": f"[REF] {name}",
        "char": name,
        "animal_std": animal_stats["std"],
        "animal_range": animal_stats["range"],
        "digit_std": digit_stats["std"],
        "digit_range": digit_stats["range"],
        "number_std": number_stats["std"],
        "number_range": number_stats["range"],
        "animal_scores": animal_stats["scores"],
        "animal_centered": animal_stats["centered_scores"],
        "digit_scores": digit_stats["scores"],
        "digit_centered": digit_stats["centered_scores"],
    })

# Add candidates
for r in candidate_results:
    c = r["candidate"]
    h = r["layer_hidden_states"][last_layer]
    animal_stats = compute_neutrality(h, animal_unembed)
    digit_stats = compute_neutrality(h, digit_unembed)
    number_stats = compute_neutrality(h, number_unembed)
    all_results.append({
        "label": f"{c['char']} {c['codepoint']}",
        "char": c["char"],
        "animal_std": animal_stats["std"],
        "animal_range": animal_stats["range"],
        "digit_std": digit_stats["std"],
        "digit_range": digit_stats["range"],
        "number_std": number_stats["std"],
        "number_range": number_stats["range"],
        "animal_scores": animal_stats["scores"],
        "animal_centered": animal_stats["centered_scores"],
        "digit_scores": digit_stats["scores"],
        "digit_centered": digit_stats["centered_scores"],
    })

# Sort by combined neutrality (low animal_std + low digit_std)
all_results.sort(key=lambda x: x["animal_std"] + x["digit_std"])

print(f"\nRanked by neutrality at L{last_layer} (lowest animal_std + digit_std first):")
print(f"{'Label':<30} {'AnimalStd':>10} {'AnimalRange':>12} {'DigitStd':>10} {'DigitRange':>12} {'NumStd':>10}")
print("-" * 100)
for r in all_results:
    print(f"{r['label']:<30} {r['animal_std']:>10.2f} {r['animal_range']:>12.2f} {r['digit_std']:>10.2f} {r['digit_range']:>12.2f} {r['number_std']:>10.2f}")

Single-token animals (12): ['cat', 'dog', 'owl', 'lion', 'wolf', 'bear', 'horse', 'rabbit', 'snake', 'fox', 'deer', 'hawk']
Skipped multi-token: ['tiger', 'eagle', 'dolphin', 'elephant', 'penguin', 'whale', 'shark', 'panther']
Animals: 12 tokens
Digits (0-9): 10 tokens
3-digit numbers (sampled): 18 tokens

Ranked by neutrality at L27 (lowest animal_std + digit_std first):
Label                           AnimalStd  AnimalRange   DigitStd   DigitRange     NumStd
----------------------------------------------------------------------------------------------------
� U+FFFD                             0.68         2.33       0.99         2.96       1.02
� U+FFFD                             0.68         2.33       0.99         2.96       1.02
� U+FFFD                             0.68         2.33       0.99         2.96       1.02
� U+FFFD                             0.68         2.33       0.99         2.96       1.02
� U+FFFD                             0.68         2.33       0.99         

### Detailed per-animal and per-digit scores

For the most neutral candidates, show the individual projection onto each animal and digit direction.
Flat = good (no preference baked in).

In [20]:
# Show top 5 most neutral + the references
# Uses mean-centered scores so you see relative preferences, not absolute biases
N_SHOW = 5

for r in all_results[:N_SHOW + len(ref_results)]:
    print(f"\n{'='*60}")
    print(f"  {r['label']}  (animal_std={r['animal_std']:.2f}, digit_std={r['digit_std']:.2f})")
    print(f"{'='*60}")

    # Per-animal scores: absolute + centered side by side
    print(f"\n  Animal projections:")
    animal_names = list(animal_token_ids.keys())
    raw = r["animal_scores"]
    ctr = r["animal_centered"]
    raw_range = (raw.max() - raw.min()).item()
    sorted_idx = raw.argsort(descending=True)
    print(f"    {'name':<12} {'abs':>7} {'centered':>9}  bar (abs)")
    for idx in sorted_idx:
        bar_len = max(0, int((raw[idx].item() - raw.min().item()) / max(raw_range, 1e-6) * 25))
        print(f"    {animal_names[idx]:<12} {raw[idx].item():>+7.2f} {ctr[idx].item():>+9.2f}  {'█' * bar_len}")

    # Per-digit scores: absolute + centered side by side
    print(f"\n  Digit projections:")
    digit_names = list(digit_token_ids.keys())
    draw = r["digit_scores"]
    dctr = r["digit_centered"]
    draw_range = (draw.max() - draw.min()).item()
    sorted_idx = draw.argsort(descending=True)
    print(f"    {'d':<4} {'abs':>7} {'centered':>9}  bar (abs)")
    for idx in sorted_idx:
        bar_len = max(0, int((draw[idx].item() - draw.min().item()) / max(draw_range, 1e-6) * 25))
        print(f"    {digit_names[idx]:<4} {draw[idx].item():>+7.2f} {dctr[idx].item():>+9.2f}  {'█' * bar_len}")


  � U+FFFD  (animal_std=0.68, digit_std=0.99)

  Animal projections:
    name             abs  centered  bar (abs)
    cat            +2.25     +1.68  █████████████████████████
    wolf           +1.38     +0.81  ███████████████
    bear           +0.85     +0.28  █████████
    lion           +0.78     +0.21  █████████
    rabbit         +0.63     +0.06  ███████
    snake          +0.30     -0.27  ████
    owl            +0.29     -0.28  ████
    deer           +0.17     -0.40  ██
    fox            +0.16     -0.41  ██
    horse          +0.12     -0.45  ██
    hawk           -0.02     -0.59  
    dog            -0.08     -0.65  

  Digit projections:
    d        abs  centered  bar (abs)
    2      +9.99     +2.09  █████████████████████████
    1      +9.32     +1.42  ███████████████████
    0      +8.08     +0.18  ████████
    3      +7.81     -0.09  ██████
    5      +7.54     -0.36  ████
    9      +7.49     -0.42  ███
    4      +7.40     -0.50  ███
    8      +7.31     -0.59  ██

### Save representations

In [ ]:
save_data = {
    "reference": {
        name: {
            "layer_hidden_states": r["layer_hidden_states"],
            "layer_logits": r["layer_logits"],
        }
        for name, r in ref_results.items()
    },
    "candidates": {
        r["candidate"]["codepoint"]: {
            "char": r["candidate"]["char"],
            "name": r["candidate"]["name"],
            "token_id": r["candidate"]["token_id"],
            "layer_hidden_states": r["layer_hidden_states"],
            "layer_logits": r["layer_logits"],
        }
        for r in candidate_results
    },
    "neutrality_ranking": [
        {"label": r["label"], "animal_std": r["animal_std"], "digit_std": r["digit_std"],
         "animal_range": r["animal_range"], "digit_range": r["digit_range"]}
        for r in all_results
    ],
}

save_path = "token_exploration_results.pt"
torch.save(save_data, save_path)
print(f"Saved to {save_path}")